# 04 — Final test evaluation and failure analysis

This notebook opens the test period once. It reads the validation result, refits
the already-selected ML model on all eligible pre-test observations, compares it
with the baselines, and diagnoses when the forecast fails.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Run this notebook from the project root or notebooks folder.")

# This fallback makes the src-layout package importable even before an editable install.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd

from market_volatility.data import (
    MODELS_DIR,
    REPORTS_DIR,
    chronological_split,
    final_training_set,
    load_modeling_dataset,
)
from market_volatility.evaluate import (
    errors_by_regime,
    forecast_metrics,
    prediction_frame,
    training_regime_thresholds,
    worst_forecasts,
)
from market_volatility.features import FORWARD_HORIZON, TARGET_COLUMN
from market_volatility.plotting import save_figure
from market_volatility.train import (
    baseline_predictions,
    fit_models,
    model_specs,
    predict_models,
)

VALIDATION_START = "2016-01-01"
TEST_START = "2021-01-01"

In [ ]:
validation_path = REPORTS_DIR / "validation_metrics.csv"
if not validation_path.exists():
    raise FileNotFoundError("Run notebook 03 before opening the test set.")

validation_metrics = pd.read_csv(validation_path, index_col="model")
candidate_names = list(model_specs())
selected_model = validation_metrics.loc[candidate_names, "MAE"].idxmin()
print(f"Model fixed before test evaluation: {selected_model}")

In [ ]:
modeling_data = load_modeling_dataset()
splits = chronological_split(
    modeling_data,
    validation_start=VALIDATION_START,
    test_start=TEST_START,
    purge_horizon=FORWARD_HORIZON,
)

# The train-validation boundary no longer matters after model selection, so use
# all observations before the test boundary, retaining only the test purge.
final_train = final_training_set(
    modeling_data,
    test_start=TEST_START,
    purge_horizon=FORWARD_HORIZON,
)
final_model = fit_models(final_train, names=[selected_model])[selected_model]

test_predictions = baseline_predictions(
    splits.test,
    training_target_mean=float(final_train[TARGET_COLUMN].mean()),
).join(predict_models({selected_model: final_model}, splits.test))

test_metrics = forecast_metrics(splits.test[TARGET_COLUMN], test_predictions)
display(test_metrics.style.format("{:.4f}"))

In [ ]:
test_table = prediction_frame(splits.test[TARGET_COLUMN], test_predictions)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
test_metrics.to_csv(REPORTS_DIR / "test_metrics.csv")
test_table.to_csv(REPORTS_DIR / "test_predictions.csv", index_label="Date")
joblib.dump(final_model, MODELS_DIR / "selected_model.joblib")
(MODELS_DIR / "selected_model.txt").write_text(f"{selected_model}\n")

## Where does the selected model fail?

We inspect three complementary views:

1. Error through time: prolonged failure and slow adaptation.
2. Error by realized-volatility regime: smoothing of spikes versus overreaction.
3. Largest individual errors: specific episodes worth investigating.

Regime cutoffs are learned from pre-test labels, avoiding test-derived bins.

In [ ]:
thresholds = training_regime_thresholds(final_train[TARGET_COLUMN])
regime_errors = errors_by_regime(
    splits.test[TARGET_COLUMN], test_predictions[selected_model], thresholds
)
worst = worst_forecasts(
    splits.test[TARGET_COLUMN], test_predictions[selected_model], count=15
)

display(regime_errors.style.format({
    "mean_actual": "{:.3f}",
    "MAE": "{:.3f}",
    "bias": "{:.3f}",
    "underprediction_rate": "{:.1%}",
}))
display(worst)

regime_errors.to_csv(REPORTS_DIR / "test_errors_by_regime.csv")
worst.to_csv(REPORTS_DIR / "worst_test_forecasts.csv", index_label="Date")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(splits.test.index, splits.test[TARGET_COLUMN], label="Actual", color="black")
ax.plot(test_predictions.index, test_predictions[selected_model], label=selected_model)
ax.set(title="Final test: actual versus forecast volatility", ylabel="Annualized volatility", xlabel="Date")
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "04_test_actual_vs_forecast")
plt.show()

In [ ]:
selected_error = test_predictions[selected_model] - splits.test[TARGET_COLUMN]
rolling_mae = selected_error.abs().rolling(60).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].plot(selected_error.index, selected_error, linewidth=0.7, color="darkred")
axes[0].set(title="Signed error: forecast minus actual", ylabel="Error")
axes[1].plot(rolling_mae.index, rolling_mae, linewidth=1.0, color="purple")
axes[1].set(title="60-session rolling MAE", ylabel="MAE", xlabel="Date")
for axis in axes:
    axis.grid(alpha=0.25)
save_figure(fig, "04_test_error_over_time")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    splits.test[TARGET_COLUMN],
    test_predictions[selected_model],
    s=14,
    alpha=0.4,
)
limits = [0, max(splits.test[TARGET_COLUMN].max(), test_predictions[selected_model].max())]
ax.plot(limits, limits, linestyle="--", color="black", label="perfect forecast")
ax.set(xlabel="Actual forward volatility", ylabel="Forecast volatility", title="Calibration on final test")
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "04_test_calibration")
plt.show()

## How to interpret failure

- Negative high-regime bias means the model smooths sudden volatility jumps; the
  predictors describe yesterday well but cannot foresee new shocks.
- Positive bias after spikes means trailing-window features decay slowly.
- A nonlinear model that fails to beat persistence probably learned historical
  regime details that did not repeat.
- Better MAE but worse QLIKE/RMSE means typical forecasts improved while extreme
  misses worsened.

The honest conclusion can be that persistence remains hard to beat. That is a
useful empirical result, not a failed project. The saved CSVs make every reported
number auditable row by row.